In [23]:
import sys
import numpy as np
import pandas as pd

print("PYTHON:", sys.executable)
print("numpy:", np.__version__, np.__file__)
print("pandas:", pd.__version__, pd.__file__)

PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
numpy: 2.2.6 /home/mgjeong/miniconda3/envs/paper-gpu/lib/python3.10/site-packages/numpy/__init__.py
pandas: 2.3.3 /home/mgjeong/miniconda3/envs/paper-gpu/lib/python3.10/site-packages/pandas/__init__.py


In [24]:
from pathlib import Path
import os
import sys
import subprocess
import json
import pandas as pd
from datetime import datetime

import torch

In [25]:
!pwd

/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/notebooks


In [26]:
#path = "/root/llm/JOILang-Server"
path = "/home/mgjeong/Desktop/llm/JOILang-Server"

In [27]:
#py_path = "/root/llm/je/bin/python"
py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

In [28]:
# =============================================================================
# 0. Kernel / Python 환경 확인
# =============================================================================
print("=" * 100)
print("0. Kernel / Python 환경 확인")
print("KERNEL PYTHON:", sys.executable)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("torch cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available in the current Jupyter kernel. "
        f"Kernel이 {py_path}인지 확인하세요."
    )

# resolve() 사용 금지: /root/llm/je/bin/python이 anaconda 원본으로 풀릴 수 있음
# /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
# a100 : "/root/llm/je/bin/python"
EXPECTED_PYTHON = os.path.abspath(py_path)
CURRENT_PYTHON = os.path.abspath(sys.executable)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Wrong Jupyter kernel Python.\n"
        f"Expected: {EXPECTED_PYTHON}\n"
        f"Current : {CURRENT_PYTHON}\n"
        f"Jupyter에서 Kernel → Change Kernel → Python (/root/llm/je)로 바꾸세요."
    )


# =============================================================================
# 1. Repository / Script path 설정
# =============================================================================
print("=" * 100)
print("1. Repository / Script path 설정")
# =========================
# Server profile
# =========================

# A100/root 서버
# path = "/root/llm/JOILang-Server"
# py_path = "/root/llm/je/bin/python"
# local_model_base = "/root/llm/local_models"

# A6000/mgjeong 서버
path = "/home/mgjeong/Desktop/llm/JOILang-Server"
py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"
local_model_base = "/home/mgjeong/Desktop/llm/local_models"

REPO = Path(path).resolve()
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"
RESULTS_ROOT = REPO / "gpt_mg/version0_15_update20260413/results"
LOCAL_MODEL_BASE = Path(local_model_base).resolve()

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

print("REPO:", REPO, REPO.exists())
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("RESULTS_ROOT:", RESULTS_ROOT)
print("PYTHON:", py_path, Path(py_path).exists())
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())


# =============================================================================
# 2. JOILang local model / worker 환경변수 설정
# =============================================================================
print("=" * 100)
print("2. JOILang local model / worker 환경변수 설정")
# 이전 실행에서 남아 있을 수 있는 충돌 변수 제거
for key in [
    "JOI_V15_LOCAL_MODEL_NAME",
    "JOI_V14_LOCAL_MODEL_NAME",
    "JOI_V14_WORKER_PYTHON",
    "JOI_V15_PERSISTENT_WORKER",   # 중요: persistent worker를 끄지 않기 위해 제거
]:
    os.environ.pop(key, None)

os.environ["PYTHONUNBUFFERED"] = "1"

# persistent worker는 유지하되, local model 위치만 지정
os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

# worker도 현재 Jupyter kernel python과 동일하게 고정
os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

# debug
os.environ["JOI_V15_DEBUG_WORKER"] = "1"
os.environ["JOI_V15_DEBUG_LOG"] = "/tmp/joi_v15_worker_debug.log"

print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))
print("JOI_V15_DEBUG_LOG:", os.environ.get("JOI_V15_DEBUG_LOG"))


# =============================================================================
# 3. subprocess에서도 CUDA가 정상인지 확인
# =============================================================================
print("3. subprocess에서도 CUDA가 정상인지 확인")
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import sys; "
            "print('subprocess python:', sys.executable); "
            "import torch; "
            "print('subprocess torch:', torch.__version__); "
            "print('subprocess cuda runtime:', torch.version.cuda); "
            "print('subprocess cuda available:', torch.cuda.is_available()); "
            "assert torch.cuda.is_available(), 'CUDA is not available in subprocess'; "
            "print('subprocess gpu:', torch.cuda.get_device_name(0))"
        ),
    ],
    check=True,
)

print("=" * 100)
print("Environment setup complete.")

0. Kernel / Python 환경 확인
KERNEL PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
pandas: 2.3.3
torch: 2.9.1+cu128
torch cuda runtime: 12.8
cuda available: True
GPU: NVIDIA RTX A6000
1. Repository / Script path 설정
REPO: /home/mgjeong/Desktop/llm/JOILang-Server True
SCRIPT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py True
RESULTS_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python True
LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models True
2. JOILang local model / worker 환경변수 설정
JOI_V15_WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_DEBUG_LOG: /tmp/joi_v15_worker_debug.log
3. subprocess에서도 CUDA가 정상인지 확인
subprocess python: /home/mgjeong/m

In [29]:
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", p)
    print("realpath:", p.resolve() if p.exists() else None)
    print("exists:", p.exists())

    if p.exists():
        real = p.resolve()
        print("config:", (p / "config.json").exists())
        print("tokenizer:", (p / "tokenizer.json").exists())
        print("index:", (p / "model.safetensors.index.json").exists())
        print("safetensors:", len(list(real.glob("*.safetensors"))))


 qwen25_coder_7b
path: /home/mgjeong/Desktop/llm/local_models/qwen25_coder_7b
realpath: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
exists: True
config: True
tokenizer: True
index: True
safetensors: 4

 llama31_8b
path: /home/mgjeong/Desktop/llm/local_models/llama31_8b
realpath: /home/mgjeong/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659
exists: True
config: True
tokenizer: True
index: True
safetensors: 4

 qwen25_coder_14b
path: /home/mgjeong/Desktop/llm/local_models/qwen25_coder_14b
realpath: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-14B-Instruct/snapshots/aedcc2d42b622764e023cf882b6652e646b95671
exists: True
config: True
tokenizer: True
index: True
safetensors: 6

 phi35_mini
path: /home/mgjeong/Desktop/llm/local_models/phi35_mini
realpath: None
exists: False

 gemma2_9b_it
path: /home/mgjeong/Desktop/llm/lo

In [30]:
def run_ga_smoke_pair(
    label: str,
    model_key: str,
    target_detpass: float,
    use_cloud_advisor: bool,
    categories=(1, 2),
    limit_per_category=2,
    population=2,
    gens=3,
    sample_size=4,
    validation_size=4,
    timeout_sec=600,
    launcher_python: str = None,
    worker_python: str = None,
    force_worker_mode: bool = True,
    advisor_trigger_mode: str = "always",
    advisor_min_population_for_child: int = 4,
    advisor_force_child_quota: bool = True,
):
    launcher_python = str(Path(launcher_python or py_path).resolve())
    worker_python = str(Path(worker_python or launcher_python).resolve())

    if not Path(launcher_python).exists():
        raise FileNotFoundError(f"launcher_python does not exist: {launcher_python}")
    if not Path(worker_python).exists():
        raise FileNotFoundError(f"worker_python does not exist: {worker_python}")
    if not SCRIPT.exists():
        raise FileNotFoundError(f"SCRIPT does not exist: {SCRIPT}")
    if model_key not in MODEL_DIRS:
        raise KeyError(f"Unknown model_key={model_key}. Available={list(MODEL_DIRS)}")

    local_model_path = (LOCAL_MODEL_BASE / MODEL_DIRS[model_key]).resolve()

    if not local_model_path.exists():
        raise FileNotFoundError(f"local model path does not exist: {local_model_path}")
    if not (local_model_path / "config.json").exists():
        raise FileNotFoundError(f"config.json not found: {local_model_path}")
    if not (local_model_path / "tokenizer.json").exists():
        raise FileNotFoundError(f"tokenizer.json not found: {local_model_path}")
    if not (local_model_path / "model.safetensors.index.json").exists():
        raise FileNotFoundError(f"model.safetensors.index.json not found: {local_model_path}")

    if use_cloud_advisor and not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Use getpass before running real cloud-advisor mode."
        )

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    cat_text = "".join(str(c) for c in categories)

    mode = "cloud_advisor" if use_cloud_advisor else "cloudless"
    worker_tag = "forcedworker" if force_worker_mode else "defaultmode"

    out_dir = (
        RESULTS_ROOT
        / f"ga_smoke_{mode}_{worker_tag}_cat{cat_text}_lpc{limit_per_category}"
          f"_pop{population}_gens{gens}_{model_key}_{ts}"
        / "ga_output"
    )

    debug_log = f"/tmp/joi_v15_worker_debug_{model_key}_{ts}.log"

    cmd = [
        launcher_python,
        "-u",
        str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),

        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),

        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "1",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--progress", "verbose",
        "--timeout-sec", str(timeout_sec),
        "--retries", "0",
        "--full-run",
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(out_dir),
    ]

    if force_worker_mode:
        cmd += ["--llm-mode", "worker"]

    for c in categories:
        cmd += ["--category", str(c)]

    if use_cloud_advisor:
        cmd += [
            "--llm-mutation-advisor",
            "--advisor-model-key", "gpt41_mini",
            "--advisor-trigger-mode", advisor_trigger_mode,
            "--advisor-min-population-for-child", str(advisor_min_population_for_child),
        ]
        if advisor_force_child_quota:
            cmd += ["--advisor-force-child-quota"]
    else:
        cmd += ["--advisor-trigger-mode", "off"]

    run_env = os.environ.copy()

    # 이전 서버에서 남은 잘못된 경로 제거
    run_env.pop("JOI_V14_WORKER_PYTHON", None)

    # 모든 경로는 위의 server profile 변수에서 파생
    run_env["JOI_V15_WORKER_PYTHON"] = worker_python
    run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
    run_env["JOI_V15_LOCAL_MODEL_NAME"] = str(local_model_path)
    run_env["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
    run_env["JOI_V15_LOCAL_FILES_ONLY"] = "true"

    # offline/local cache 사용 강제
    run_env["TRANSFORMERS_OFFLINE"] = "1"
    run_env["HF_HUB_OFFLINE"] = "1"

    # torch backend 명시
    run_env["USE_TORCH"] = "1"
    run_env["USE_TF"] = "0"
    run_env["USE_FLAX"] = "0"
    run_env.pop("TRANSFORMERS_NO_TORCH", None)

    # debug
    run_env["JOI_V15_DEBUG_WORKER"] = "1"
    run_env["JOI_V15_DEBUG_LOG"] = debug_log

    print("=" * 100)
    print(f"RUN: {label} / {model_key} / {mode}_{worker_tag}")
    print("OUTPUT:", out_dir)
    print("REPO:", REPO)
    print("KERNEL_PYTHON:", sys.executable)
    print("LAUNCHER_PYTHON:", launcher_python)
    print("WORKER_PYTHON:", worker_python)
    print("FORCE_WORKER_MODE:", force_worker_mode)
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"])
    print("JOI_V15_LOCAL_MODEL_NAME:", run_env["JOI_V15_LOCAL_MODEL_NAME"])
    print("JOI_V15_LOCAL_MODEL_REALPATH:", local_model_path)
    print("JOI_V15_LOCAL_DEVICE:", run_env["JOI_V15_LOCAL_DEVICE"])
    print("JOI_V15_LOCAL_FILES_ONLY:", run_env["JOI_V15_LOCAL_FILES_ONLY"])
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(cmd))
    print("=" * 100)

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()
    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(f"{label} {mode}_{worker_tag} failed with return code {rc}")

    return out_dir, debug_log

In [31]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_worker_cat1_sample1_localpath",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1,),
    limit_per_category=1,
    population=1,
    gens=1,
    sample_size=1,
    validation_size=1,
    timeout_sec=600,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=True,
)

RUN: smoke_worker_cat1_sample1_localpath / qwen25_coder_7b / cloudless_forcedworker
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_forcedworker_cat1_lpc1_pop1_gens1_qwen25_coder_7b_20260602_224259/ga_output
REPO: /home/mgjeong/Desktop/llm/JOILang-Server
KERNEL_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
LAUNCHER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
FORCE_WORKER_MODE: True
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_MODEL_NAME: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_MODEL_REALPATH: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
DEBUG_LOG:

In [32]:
import subprocess

code = r'''
import sys
print("python:", sys.executable)

for m in ["torch", "transformers", "accelerate", "safetensors", "huggingface_hub"]:
    try:
        mod = __import__(m)
        print(m, "OK", getattr(mod, "__version__", ""))
    except Exception as e:
        print(m, "FAIL", repr(e))

import torch
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device0:", torch.cuda.get_device_name(0))

from transformers.utils import is_torch_available
print("transformers.is_torch_available:", is_torch_available())

from transformers import AutoTokenizer, AutoModelForCausalLM
print("AutoModelForCausalLM import OK")
'''

print(subprocess.check_output(
    [py_path, "-c", code],
    text=True,
    stderr=subprocess.STDOUT,
))

python: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
torch OK 2.9.1+cu128
transformers OK 4.57.3
accelerate OK 1.12.0
safetensors OK 0.7.0
huggingface_hub OK 0.36.0
cuda available: True
cuda device count: 2
device0: NVIDIA RTX A6000
transformers.is_torch_available: True
AutoModelForCausalLM import OK



#### debug log를 확인

In [33]:
from pathlib import Path

p = Path(debug_log)
print("debug_log:", p)
print("exists:", p.exists())

if p.exists():
    text = p.read_text(encoding="utf-8", errors="replace")
    print(text[-10000:])

debug_log: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_224259.log
exists: True

[2026-06-02T22:43:02] START persistent worker
worker_python=/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
worker_path=/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/utils/persistent_qwen_worker.py
cwd=/home/mgjeong/Desktop/llm/JOILang-Server
JOI_V15_LOCAL_DEVICE=cuda:0
TRANSFORMERS_VERBOSITY=error
HF_HUB_DISABLE_PROGRESS_BARS=1
TOKENIZERS_PARALLELISM=false
PYTHONUNBUFFERED=1
PYTHONFAULTHANDLER=1
payload_summary={"keys": ["local_attn_implementation", "local_device", "local_dtype", "local_files_only", "local_hf_modules_cache", "local_load_in_4bit", "local_max_new_tokens", "local_model_name", "local_trust_remote_code", "messages", "model"], "model": "Qwen/Qwen2.5-Coder-7B-Instruct", "local_model_name": "/home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242", "local_device": "cuda:0", "local_dty

In [34]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat1_sample2",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1,),
    limit_per_category=2,
    population=2,
    gens=1,
    sample_size=2,
    validation_size=2,
    timeout_sec=600,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat1_sample2 / qwen25_coder_7b / cloudless_defaultmode
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_defaultmode_cat1_lpc2_pop2_gens1_qwen25_coder_7b_20260602_224453/ga_output
REPO: /home/mgjeong/Desktop/llm/JOILang-Server
KERNEL_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
LAUNCHER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_MODEL_NAME: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_MODEL_REALPATH: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
DEBUG_LOG: /tmp/jo

## 2단계: category 1,2 smoke

In [35]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat12",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1, 2),
    limit_per_category=2,
    population=2,
    gens=2,
    sample_size=4,
    validation_size=4,
    timeout_sec=900,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat12 / qwen25_coder_7b / cloudless_defaultmode
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_defaultmode_cat12_lpc2_pop2_gens2_qwen25_coder_7b_20260602_224948/ga_output
REPO: /home/mgjeong/Desktop/llm/JOILang-Server
KERNEL_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
LAUNCHER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_MODEL_NAME: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_MODEL_REALPATH: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
DEBUG_LOG: /tmp/joi_v15_

## 3단계: 기존 GA 설정에 가깝게 확대

In [36]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat12_pop5_gens3",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1, 2),
    limit_per_category=3,
    population=5,
    gens=3,
    sample_size=6,
    validation_size=6,
    timeout_sec=1800,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat12_pop5_gens3 / qwen25_coder_7b / cloudless_defaultmode
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_defaultmode_cat12_lpc3_pop5_gens3_qwen25_coder_7b_20260602_230130/ga_output
REPO: /home/mgjeong/Desktop/llm/JOILang-Server
KERNEL_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
LAUNCHER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_MODEL_NAME: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_MODEL_REALPATH: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-7B-Instruct/snapshots/c03e6d358207e414f1eca0bb1891e29f1db0e242
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
DEBUG_LOG: /t

# 3. 본격 테스트
## “환경 설정 → 실행 wrapper → 6개 run 실행 → 요약/Delta 생성”
A6000 SetA = 중간 규모, 안정성/비용/시간 균형

A100  SetB = 원래 full cloudless 조건에 가까운 본 실험 규모



## API key 입력 셀

In [17]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## Cell 1. 기본 경로 / 환경 / 서버 preset 설정


In [37]:
from pathlib import Path
import os
import sys
import json
import time
import subprocess
import traceback
from datetime import datetime

import pandas as pd


# =============================================================================
# 0. Server preset 선택
# =============================================================================

# A6000 서버에서는 이 값 사용
SERVER_PRESET = "A6000_SET_A"

# A100 서버에서는 위 줄을 주석 처리하고 아래 줄 사용
#SERVER_PRESET = "A100_SET_B"


# =============================================================================
# 1. Repository / script path 설정
# =============================================================================

try:
    REPO = Path(path).absolute()
except NameError:
    REPO = Path.cwd().absolute()

VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = REPO / "local_models"

assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SERVER_PRESET:", SERVER_PRESET)
print("REPO:", REPO)
print("SCRIPT:", SCRIPT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("LOCAL_MODEL_BASE exists:", LOCAL_MODEL_BASE.exists())
print("PYTHON:", sys.executable)


# =============================================================================
# 2. 모델 목록
# =============================================================================

MODEL_LIST = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

RUN_MODES = [
    ("cloudless", False),
    ("cloud_advisor", True),
]


# =============================================================================
# 3. 서버별 실험 설정
# =============================================================================

# A6000: 중간 규모. 48GB급에서 14B까지 안정적으로 비교하기 위한 SetA.
SET_A_A6000 = dict(
    target_detpass=90,
    categories=range(1, 9),
    limit_per_category=2,
    sample_size=16,
    validation_size=16,
    population=4,
    gens=6,
    full_run=True,
    progress="verbose",
    timeout_sec=900,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=24 * 3600,
)

# A100: 기존 cloudless full 설정에 가까운 본 비교 SetB.
SET_B_A100 = dict(
    target_detpass=90,
    categories=range(1, 9),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    full_run=True,
    progress="verbose",
    timeout_sec=1200,
    retries=0,
    idle_timeout_sec=3600,
    total_timeout_sec=40 * 3600,
)

if SERVER_PRESET == "A6000_SET_A":
    COMMON_GA_CONFIG = SET_A_A6000
elif SERVER_PRESET == "A100_SET_B":
    COMMON_GA_CONFIG = SET_B_A100
else:
    raise ValueError(f"Unknown SERVER_PRESET: {SERVER_PRESET}")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
COMPARISON_ROOT = RESULTS_ROOT / f"fair_compare_{SERVER_PRESET}_{RUN_TAG}"
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("COMMON_GA_CONFIG:", COMMON_GA_CONFIG)
print("COMPARISON_ROOT:", COMPARISON_ROOT)


# =============================================================================
# 4. 환경변수 설정
# =============================================================================

def setup_env_for_current_server():
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

    # 공통 충돌 변수 제거
    os.environ.pop("JOI_V15_LOCAL_MODEL_NAME", None)
    os.environ.pop("JOI_V14_LOCAL_MODEL_NAME", None)
    os.environ.pop("JOI_V14_WORKER_PYTHON", None)

    # 중요: persistent worker를 끄면 안 됨.
    # false가 남아 있으면 version0_13/qwen_local_worker.py fallback 가능.
    os.environ.pop("JOI_V15_PERSISTENT_WORKER", None)

    if SERVER_PRESET == "A100_SET_B":
        # A100에서는 offline local model 경로를 명확히 고정
        os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
        os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
        os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

    elif SERVER_PRESET == "A6000_SET_A":
        # A6000에서 기존 성공 환경을 최대한 보존.
        # 이미 잘 도는 경우 LOCAL_MODEL_BASE_DIR을 강제하지 않음.
        # 필요할 때만 아래 3줄을 켜면 됨.
        # os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
        # os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
        # os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
        pass

    print("=" * 100)
    print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))

setup_env_for_current_server()


# =============================================================================
# 5. Cloud advisor API key 확인
# =============================================================================

if not os.environ.get("OPENAI_API_KEY"):
    print("[WARN] OPENAI_API_KEY is not set. cloud_advisor runs will fail unless set.")
else:
    print("[OK] OPENAI_API_KEY is set.")

SERVER_PRESET: A6000_SET_A
REPO: /home/mgjeong/Desktop/llm/JOILang-Server
SCRIPT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py
RESULTS_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/JOILang-Server/local_models
LOCAL_MODEL_BASE exists: False
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
COMMON_GA_CONFIG: {'target_detpass': 90, 'categories': range(1, 9), 'limit_per_category': 2, 'sample_size': 16, 'validation_size': 16, 'population': 4, 'gens': 6, 'full_run': True, 'progress': 'verbose', 'timeout_sec': 900, 'retries': 0, 'idle_timeout_sec': 2400, 'total_timeout_sec': 86400}
COMPARISON_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A6000_SET_A_20260602_235106
JOI_V15_WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/

## Cell 2. run_ga_all_categories wrapper 정의


In [38]:
# =============================================================================
# run_ga_search.py CLI wrapper
# =============================================================================

def _get_run_ga_help_text():
    try:
        proc = subprocess.run(
            [sys.executable, str(SCRIPT), "--help"],
            cwd=str(REPO),
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=60,
        )
        return proc.stdout or ""
    except Exception as e:
        print("[WARN] failed to inspect --help:", repr(e))
        return ""

_RUN_GA_HELP_TEXT = _get_run_ga_help_text()

def _has_flag(flag: str) -> bool:
    if not _RUN_GA_HELP_TEXT:
        return True
    return flag in _RUN_GA_HELP_TEXT

def _append_optional_flag(cmd, flag, value=None):
    if _has_flag(flag):
        cmd.append(flag)
        if value is not None:
            cmd.append(str(value))
    else:
        print(f"[SKIP unsupported flag] {flag}")


def run_ga_all_categories(
    model_key: str,
    categories=range(1, 9),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    target_detpass=90,
    base_prefix="ga_final",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=32 * 3600,

    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,

    launcher_python=None,
):
    if launcher_python is None:
        launcher_python = sys.executable

    # resolve() 금지. symlink가 conda 원본으로 풀리는 문제 방지.
    launcher_python = os.path.abspath(os.path.expanduser(launcher_python))

    if not Path(launcher_python).exists():
        raise FileNotFoundError(f"launcher_python does not exist: {launcher_python}")

    if use_advisor and not use_mock_advisor:
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY is not set for real cloud advisor mode.")

    categories = tuple(categories)
    cat_text = "".join(str(c) for c in categories)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    mode = "cloud_advisor" if use_advisor else "cloudless"
    mock_tag = "_mock" if use_advisor and use_mock_advisor else ""
    safe_model_key = model_key.replace("/", "_").replace(":", "_")

    run_name = (
        f"{base_prefix}_{mode}{mock_tag}_cat{cat_text}"
        f"_lpc{limit_per_category}_pop{population}_gens{gens}"
        f"_{safe_model_key}_{ts}"
    )

    out_dir = RESULTS_ROOT / run_name / "ga_output"

    cmd = [
        launcher_python,
        "-u",
        str(SCRIPT),

        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),

        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),

        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--progress", str(progress),
        "--timeout-sec", str(timeout_sec),
        "--retries", str(retries),
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(out_dir),
    ]

    _append_optional_flag(cmd, "--idle-timeout-sec", idle_timeout_sec)
    _append_optional_flag(cmd, "--total-timeout-sec", total_timeout_sec)

    if full_run:
        _append_optional_flag(cmd, "--full-run")

    for c in categories:
        cmd += ["--category", str(c)]

    if use_advisor:
        cmd += [
            "--llm-mutation-advisor",
            "--advisor-model-key", "gpt41_mini",
            "--advisor-trigger-mode", advisor_trigger_mode,
            "--advisor-min-population-for-child", str(advisor_min_population_for_child),
        ]

        if advisor_force_child_quota:
            cmd += ["--advisor-force-child-quota"]

        if use_mock_advisor:
            cmd += ["--llm-mode", "mock"]
    else:
        cmd += ["--advisor-trigger-mode", "off"]

    run_env = os.environ.copy()
    run_env["PYTHONUNBUFFERED"] = "1"
    run_env["JOI_V15_WORKER_PYTHON"] = launcher_python

    # 매우 중요: persistent worker를 끄지 않는다.
    if run_env.get("JOI_V15_PERSISTENT_WORKER") == "false":
        run_env.pop("JOI_V15_PERSISTENT_WORKER", None)

    debug_log = f"/tmp/joi_v15_worker_debug_{safe_model_key}_{mode}_{ts}.log"
    run_env["JOI_V15_DEBUG_WORKER"] = "1"
    run_env["JOI_V15_DEBUG_LOG"] = debug_log

    print("=" * 120)
    print(f"RUN: {model_key} / {mode}{mock_tag}")
    print("OUTPUT:", out_dir)
    print("LAUNCHER_PYTHON:", launcher_python)
    print("JOI_V15_WORKER_PYTHON:", run_env.get("JOI_V15_WORKER_PYTHON"))
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", run_env.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", run_env.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", run_env.get("JOI_V15_PERSISTENT_WORKER"))
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(cmd))
    print("=" * 120)

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert proc.stdout is not None

    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")

    return out_dir

## Cell 3. 3개 모델 × 2조건 자동 실행

### 본격 6개 run 전에 반드시 1회만 테스트

In [39]:
sanity_out = run_ga_all_categories(
    model_key="qwen25_coder_7b",
    categories=range(1, 3),
    limit_per_category=1,
    sample_size=2,
    validation_size=2,
    population=2,
    gens=1,
    target_detpass=90,
    base_prefix=f"sanity_{SERVER_PRESET}",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
)

sanity_out

[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_7b / cloudless
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/sanity_A6000_SET_A_cloudless_cat12_lpc1_pop2_gens1_qwen25_coder_7b_20260602_235106/ga_output
LAUNCHER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
JOI_V15_WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_cloudless_20260602_235106.log
COMMAND:
/home/mgjeong/miniconda3/envs/paper-gpu/bin/python -u /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 90 --population 2 --gens 1 --min-generations 1 --max-generations 1 --sample-size

PosixPath('/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/sanity_A6000_SET_A_cloudless_cat12_lpc1_pop2_gens1_qwen25_coder_7b_20260602_235106/ga_output')

In [21]:
!pwd

/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/notebooks


In [40]:
ga_runs_fair = {}

for label, model_key in MODEL_LIST:
    for mode_name, use_advisor in RUN_MODES:
        run_key = f"{label}_{mode_name}"

        print("\n" + "#" * 120)
        print(f"START RUN: {run_key} / {model_key}")
        print("#" * 120)

        try:
            out_dir = run_ga_all_categories(
                model_key=model_key,
                categories=COMMON_GA_CONFIG["categories"],
                limit_per_category=COMMON_GA_CONFIG["limit_per_category"],
                sample_size=COMMON_GA_CONFIG["sample_size"],
                validation_size=COMMON_GA_CONFIG["validation_size"],
                population=COMMON_GA_CONFIG["population"],
                gens=COMMON_GA_CONFIG["gens"],
                target_detpass=COMMON_GA_CONFIG["target_detpass"],
                base_prefix=f"ga_{SERVER_PRESET}",
                use_advisor=use_advisor,
                full_run=COMMON_GA_CONFIG["full_run"],
                progress=COMMON_GA_CONFIG["progress"],
                timeout_sec=COMMON_GA_CONFIG["timeout_sec"],
                retries=COMMON_GA_CONFIG["retries"],
                idle_timeout_sec=COMMON_GA_CONFIG["idle_timeout_sec"],
                total_timeout_sec=COMMON_GA_CONFIG["total_timeout_sec"],

                advisor_trigger_mode="always",
                advisor_min_population_for_child=4,
                advisor_force_child_quota=True,
                use_mock_advisor=False,
            )

            ga_runs_fair[run_key] = out_dir
            print(f"[PASS] {run_key}: {out_dir}")

        except Exception as e:
            print(f"[FAIL] {run_key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            ga_runs_fair[run_key] = None

        time.sleep(5)


valid_ga_runs_fair = {
    key: path
    for key, path in ga_runs_fair.items()
    if path is not None
}

valid_ga_runs_fair


########################################################################################################################
START RUN: 7B_cloudless / qwen25_coder_7b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_7b / cloudless
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A6000_SET_A_cloudless_cat12345678_lpc2_pop4_gens6_qwen25_coder_7b_20260602_235347/ga_output
LAUNCHER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
JOI_V15_WORKER_PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /home/mgjeong/Desktop/llm/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_cloudless_20260602_235347.log
COMMAND:
/home/mgjeong/

{'7B_cloudless': PosixPath('/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A6000_SET_A_cloudless_cat12345678_lpc2_pop4_gens6_qwen25_coder_7b_20260602_235347/ga_output'),
 '7B_cloud_advisor': PosixPath('/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A6000_SET_A_cloud_advisor_cat12345678_lpc2_pop4_gens6_qwen25_coder_7b_20260603_055408/ga_output'),
 '8B_cloudless': PosixPath('/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A6000_SET_A_cloudless_cat12345678_lpc2_pop4_gens6_llama31_8b_20260603_115643/ga_output'),
 '8B_cloud_advisor': PosixPath('/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A6000_SET_A_cloud_advisor_cat12345678_lpc2_pop4_gens6_llama31_8b_20260603_192328/ga_output'),
 '14B_cloudless': PosixPath('/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A6000_SET_A_cloudless_cat12345678_lpc2_pop4

## Cell 4. 결과 요약표 생성


In [41]:
def load_json_safe(path: Path, default=None):
    if default is None:
        default = {}
    try:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    except Exception:
        pass
    return default


def read_csv_safe(path: Path):
    try:
        if path.exists() and path.stat().st_size > 0:
            return pd.read_csv(path)
    except Exception:
        pass
    return pd.DataFrame()


def summarize_ga_run(run_key: str, out_dir):
    out_dir = Path(out_dir)

    summary = load_json_safe(out_dir / "ga_summary.json", default={})
    progress_df = read_csv_safe(out_dir / "ga_generation_progress.csv")
    diag_df = read_csv_safe(out_dir / "ga_population_diagnostics.csv")

    last_progress = {}
    if not progress_df.empty:
        last_progress = progress_df.tail(1).iloc[0].to_dict()

    failure_histogram = None
    if not diag_df.empty and "failure_histogram" in diag_df.columns:
        failure_histogram = diag_df.tail(1)["failure_histogram"].iloc[0]

    label, mode = run_key.split("_", 1)

    return {
        "server_preset": SERVER_PRESET,
        "run_key": run_key,
        "label": label,
        "mode": mode,
        "out_dir": str(out_dir),

        "target_detpass": COMMON_GA_CONFIG["target_detpass"],
        "categories": ",".join(map(str, COMMON_GA_CONFIG["categories"])),
        "limit_per_category": COMMON_GA_CONFIG["limit_per_category"],
        "sample_size": COMMON_GA_CONFIG["sample_size"],
        "validation_size": COMMON_GA_CONFIG["validation_size"],
        "population": COMMON_GA_CONFIG["population"],
        "gens": COMMON_GA_CONFIG["gens"],

        "stage": summary.get("stage"),
        "stop_reason": summary.get("stop_reason"),
        "best_genome_id": summary.get("best_genome_id"),
        "best_generation": summary.get("best_generation"),

        "best_DETPass": summary.get("best_DETPass"),
        "best_avg_DET": summary.get("best_avg_DET"),
        "accepted_best_DETPass": summary.get("accepted_best_DETPass"),
        "accepted_best_avg_DET": summary.get("accepted_best_avg_DET"),
        "accepted_best_tokens": summary.get("accepted_best_tokens"),

        "compact_best_eligible": summary.get("compact_best_eligible"),
        "compact_best_DETPass": summary.get("compact_best_DETPass"),
        "compact_best_tokens": summary.get("compact_best_tokens"),

        "advisor_status": summary.get("advisor_status"),
        "advisor_used": summary.get("advisor_used"),
        "advisor_proposals_generated": summary.get("advisor_proposals_generated"),
        "advisor_proposals_accepted_applied": summary.get("advisor_proposals_accepted_applied"),
        "advisor_proposals_rejected": summary.get("advisor_proposals_rejected"),
        "advisor_children_scheduled": summary.get("advisor_children_scheduled"),

        "cloudless_mutation_used": summary.get("cloudless_mutation_used"),
        "pareto_archive_size": summary.get("pareto_archive_size"),

        "last_progress_fitness": last_progress.get("fitness"),
        "last_progress_avg_det_score": last_progress.get("avg_det_score"),
        "last_progress_det_pass_rate": last_progress.get("det_pass_rate"),
        "last_progress_advisor_triggered": last_progress.get("advisor_triggered"),

        "failure_histogram": failure_histogram,

        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
    }


summary_rows = []

for run_key, out_dir in ga_runs_fair.items():
    if out_dir is None:
        label, mode = run_key.split("_", 1)
        summary_rows.append({
            "server_preset": SERVER_PRESET,
            "run_key": run_key,
            "label": label,
            "mode": mode,
            "run_status": "failed",
            "out_dir": None,
        })
    else:
        row = summarize_ga_run(run_key, out_dir)
        row["run_status"] = "success"
        summary_rows.append(row)

fair_summary_df = pd.DataFrame(summary_rows)

summary_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_fair_summary.csv"
fair_summary_df.to_csv(summary_csv, index=False)

print("summary_csv:", summary_csv)
display(fair_summary_df)

summary_csv: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A6000_SET_A_20260602_235106/A6000_SET_A_fair_summary.csv


,server_preset,run_key,label,mode,out_dir,target_detpass,categories,limit_per_category,sample_size,validation_size,...,cloudless_mutation_used,pareto_archive_size,last_progress_fitness,last_progress_avg_det_score,last_progress_det_pass_rate,last_progress_advisor_triggered,failure_histogram,summary_exists,progress_exists,run_status
0,A6000_SET_A,7B_cloudless,7B,cloudless,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,90,"1,2,3,4,5,6,7,8",2,16,16,...,True,7,86.115458,87.4388,68.75,False,"{""extraneous"": 3, ""gt_mismatch"": 8, ""gt_receiv...",True,True,success
1,A6000_SET_A,7B_cloud_advisor,7B,cloud_advisor,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,90,"1,2,3,4,5,6,7,8",2,16,16,...,True,7,86.115458,87.4388,68.75,False,"{""extraneous"": 3, ""gt_mismatch"": 8, ""gt_receiv...",True,True,success
2,A6000_SET_A,8B_cloudless,8B,cloudless,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,90,"1,2,3,4,5,6,7,8",2,16,16,...,True,8,98.713594,88.1192,81.25,False,"{""extraneous"": 5, ""gt_mismatch"": 8, ""gt_receiv...",True,True,success
3,A6000_SET_A,8B_cloud_advisor,8B,cloud_advisor,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,90,"1,2,3,4,5,6,7,8",2,16,16,...,True,9,111.733214,90.9550,93.75,False,"{""extraneous"": 4, ""gt_mismatch"": 8, ""gt_receiv...",True,True,success
4,A6000_SET_A,14B_cloudless,14B,cloudless,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,90,"1,2,3,4,5,6,7,8",2,16,16,...,True,5,89.261937,74.6749,75.00,False,"{""extraneous"": 4, ""gt_mismatch"": 8, ""semantic""...",True,True,success
5,A6000_SET_A,14B_cloud_advisor,14B,cloud_advisor,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,90,"1,2,3,4,5,6,7,8",2,16,16,...,True,7,89.261937,74.6749,75.00,False,"{""extraneous"": 4, ""gt_mismatch"": 8, ""semantic""...",True,True,success


## Cell 5. cloudless vs cloud-advisor delta 표 생성


In [42]:
def make_delta_table(fair_summary_df):
    df = fair_summary_df.copy()
    df = df[df["run_status"] == "success"].copy()

    rows = []

    for label in ["7B", "8B", "14B"]:
        c = df[(df["label"] == label) & (df["mode"] == "cloudless")]
        a = df[(df["label"] == label) & (df["mode"] == "cloud_advisor")]

        if c.empty or a.empty:
            rows.append({
                "server_preset": SERVER_PRESET,
                "label": label,
                "status": "missing_pair",
            })
            continue

        c = c.iloc[0]
        a = a.iloc[0]

        row = {
            "server_preset": SERVER_PRESET,
            "label": label,
            "status": "ok",

            "cloudless_best_DETPass": c.get("best_DETPass"),
            "advisor_best_DETPass": a.get("best_DETPass"),
            "delta_best_DETPass": None,

            "cloudless_best_avg_DET": c.get("best_avg_DET"),
            "advisor_best_avg_DET": a.get("best_avg_DET"),
            "delta_best_avg_DET": None,

            "cloudless_tokens": c.get("accepted_best_tokens"),
            "advisor_tokens": a.get("accepted_best_tokens"),
            "delta_tokens": None,

            "cloudless_best_generation": c.get("best_generation"),
            "advisor_best_generation": a.get("best_generation"),

            "advisor_used": a.get("advisor_used"),
            "advisor_proposals_generated": a.get("advisor_proposals_generated"),
            "advisor_proposals_accepted_applied": a.get("advisor_proposals_accepted_applied"),
            "advisor_children_scheduled": a.get("advisor_children_scheduled"),

            "cloudless_out_dir": c.get("out_dir"),
            "advisor_out_dir": a.get("out_dir"),
        }

        for out_key, adv_key, base_key in [
            ("delta_best_DETPass", "advisor_best_DETPass", "cloudless_best_DETPass"),
            ("delta_best_avg_DET", "advisor_best_avg_DET", "cloudless_best_avg_DET"),
            ("delta_tokens", "advisor_tokens", "cloudless_tokens"),
        ]:
            try:
                row[out_key] = row[adv_key] - row[base_key]
            except Exception:
                row[out_key] = None

        rows.append(row)

    delta_df = pd.DataFrame(rows)

    delta_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_cloudless_vs_advisor_delta.csv"
    delta_df.to_csv(delta_csv, index=False)

    print("delta_csv:", delta_csv)
    display(delta_df)

    return delta_df


fair_delta_df = make_delta_table(fair_summary_df)

delta_csv: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A6000_SET_A_20260602_235106/A6000_SET_A_cloudless_vs_advisor_delta.csv


,server_preset,label,status,cloudless_best_DETPass,advisor_best_DETPass,delta_best_DETPass,cloudless_best_avg_DET,advisor_best_avg_DET,delta_best_avg_DET,cloudless_tokens,advisor_tokens,delta_tokens,cloudless_best_generation,advisor_best_generation,advisor_used,advisor_proposals_generated,advisor_proposals_accepted_applied,advisor_children_scheduled,cloudless_out_dir,advisor_out_dir
0,A6000_SET_A,7B,ok,68.75,68.75,0.0,87.4388,87.4388,0.0000,68677.000,68677.000,0.0,6,6,True,22,0,0,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
1,A6000_SET_A,8B,ok,81.25,93.75,12.5,90.5895,90.9550,0.3655,69483.625,69499.625,16.0,,4,True,22,4,4,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
2,A6000_SET_A,14B,ok,75.00,75.00,0.0,74.6749,74.6749,0.0000,36582.000,36582.000,0.0,3,1,True,16,4,4,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...


## Cell 6. advisor activity 확인

In [43]:
def inspect_advisor_activity(ga_runs_fair):
    rows = []

    for run_key, out_dir in ga_runs_fair.items():
        if out_dir is None:
            continue

        out_dir = Path(out_dir)

        advisor_feedback = out_dir / "advisor_feedback_batches.jsonl"
        advisor_proposals = out_dir / "advisor_mutation_proposals.jsonl"
        advisor_summary = out_dir / "advisor_mutation_summary.csv"

        rows.append({
            "server_preset": SERVER_PRESET,
            "run_key": run_key,
            "out_dir": str(out_dir),
            "advisor_feedback_exists": advisor_feedback.exists(),
            "advisor_feedback_bytes": advisor_feedback.stat().st_size if advisor_feedback.exists() else 0,
            "advisor_proposals_exists": advisor_proposals.exists(),
            "advisor_proposals_bytes": advisor_proposals.stat().st_size if advisor_proposals.exists() else 0,
            "advisor_summary_exists": advisor_summary.exists(),
            "advisor_summary_bytes": advisor_summary.stat().st_size if advisor_summary.exists() else 0,
        })

    advisor_df = pd.DataFrame(rows)

    advisor_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_advisor_activity.csv"
    advisor_df.to_csv(advisor_csv, index=False)

    print("advisor_csv:", advisor_csv)
    display(advisor_df)

    return advisor_df


advisor_activity_df = inspect_advisor_activity(ga_runs_fair)

advisor_csv: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A6000_SET_A_20260602_235106/A6000_SET_A_advisor_activity.csv


,server_preset,run_key,out_dir,advisor_feedback_exists,advisor_feedback_bytes,advisor_proposals_exists,advisor_proposals_bytes,advisor_summary_exists,advisor_summary_bytes
0,A6000_SET_A,7B_cloudless,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,False,0,True,0,True,33
1,A6000_SET_A,7B_cloud_advisor,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,True,267144,True,21630,True,8357
2,A6000_SET_A,8B_cloudless,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,False,0,True,0,True,33
3,A6000_SET_A,8B_cloud_advisor,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,True,240027,True,24822,True,11577
4,A6000_SET_A,14B_cloudless,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,False,0,True,0,True,33
5,A6000_SET_A,14B_cloud_advisor,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,True,211784,True,18019,True,8510


In [57]:
advisor_activity_df["out_dir"].

<bound method Series.view of 0    /home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
1    /home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
2    /home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
3    /home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
4    /home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
Name: out_dir, dtype: object>

In [65]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. Load A6000 result CSVs
# ============================================================

CSV_FAIR_SUMMARY = Path("/mnt/data/A6000_SET_A_fair_summary(1).csv")
CSV_DELTA = Path("/mnt/data/A6000_SET_A_cloudless_vs_advisor_delta(1).csv")
CSV_ADVISOR = Path("/mnt/data/A6000_SET_A_advisor_activity(1).csv")

fair = pd.read_csv(CSV_FAIR_SUMMARY)
delta = pd.read_csv(CSV_DELTA)
advisor = pd.read_csv(CSV_ADVISOR)

print("fair:", fair.shape)
print("delta:", delta.shape)
print("advisor:", advisor.shape)

def num(x):
    return pd.to_numeric(x, errors="coerce")

def first_existing(df, candidates, default=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series([default] * len(df), index=df.index)

def normalize_sdet(x):
    x = num(x)
    return np.where(x > 1.0, x / 100.0, x)

# ============================================================
# 2. Run health table
# ============================================================

health = fair.copy()

health["tokens_gt_0"] = num(first_existing(health, [
    "accepted_best_tokens", "avg_prompt_tokens", "best_tokens"
], 0)) > 0

health["detpass_gt_0"] = num(first_existing(health, [
    "best_DETPass", "best_so_far_DETPass", "validation_det_pass_rate"
], 0)) > 0

health["target_reached"] = (
    num(first_existing(health, ["best_DETPass", "best_so_far_DETPass"], 0))
    >= num(first_existing(health, ["target_detpass"], 90))
)

health["run_ok"] = (
    (first_existing(health, ["run_status"], "success").astype(str).str.lower().isin(["success", "ok", "true"]))
    & health["tokens_gt_0"]
    & health["detpass_gt_0"]
)

health_cols = [
    "run_key", "label", "mode", "run_status",
    "best_DETPass", "best_avg_DET", "accepted_best_tokens",
    "target_detpass", "target_reached",
    "advisor_used", "advisor_proposals_generated",
    "advisor_proposals_accepted_applied", "advisor_children_scheduled",
    "pareto_archive_size", "tokens_gt_0", "detpass_gt_0", "run_ok",
]

health_table = health[[c for c in health_cols if c in health.columns]]
display(health_table)

# ============================================================
# 3. Paper Table-4 style table
# ============================================================

model_name_map = {
    "7B": "Qwen2.5-Coder-7B",
    "8B": "Llama-3.1-8B",
    "14B": "Qwen2.5-Coder-14B",
}

config_name_map = {
    "cloudless": "GPS-PromptOps",
    "cloud_advisor": "GPS-PromptOps + Cloud Advisor",
}

paper = pd.DataFrame({
    "Model": fair["label"].map(model_name_map).fillna(fair["label"].astype(str)),
    "Configuration": fair["mode"].map(config_name_map).fillna(fair["mode"].astype(str)),
    "DETPass (%)": num(first_existing(fair, ["best_DETPass", "best_so_far_DETPass"])),
    "Avg S_DET": normalize_sdet(first_existing(fair, ["best_avg_DET", "sdet", "avg_sdet"])),
    "ReplayFit (%)": num(first_existing(fair, ["ReplayFit", "replay_fit", "replay_pass_rate"])),
    "Avg Input Tokens": num(first_existing(fair, ["accepted_best_tokens", "avg_prompt_tokens", "best_tokens"])),
    "Latency (s)": num(first_existing(fair, ["latency_sec", "warm_latency_p50", "avg_latency_sec"])),
    "Best Generation": num(first_existing(fair, ["best_generation"])),
    "Pareto Archive": num(first_existing(fair, ["pareto_archive_size"])),
    "Advisor Proposals": num(first_existing(fair, ["advisor_proposals_generated"], 0)),
    "Advisor Applied": num(first_existing(fair, ["advisor_proposals_accepted_applied", "advisor_children_scheduled"], 0)),
})

avg_rows = []
for cfg, sub in paper.groupby("Configuration"):
    avg_rows.append({
        "Model": "Target-model average",
        "Configuration": cfg,
        "DETPass (%)": sub["DETPass (%)"].mean(),
        "Avg S_DET": sub["Avg S_DET"].mean(),
        "ReplayFit (%)": sub["ReplayFit (%)"].mean(),
        "Avg Input Tokens": sub["Avg Input Tokens"].mean(),
        "Latency (s)": sub["Latency (s)"].mean(),
        "Best Generation": np.nan,
        "Pareto Archive": sub["Pareto Archive"].mean(),
        "Advisor Proposals": sub["Advisor Proposals"].sum(),
        "Advisor Applied": sub["Advisor Applied"].sum(),
    })

paper_table = pd.concat([paper, pd.DataFrame(avg_rows)], ignore_index=True)

display(
    paper_table.style.format({
        "DETPass (%)": "{:.2f}",
        "Avg S_DET": "{:.4f}",
        "ReplayFit (%)": "{:.2f}",
        "Avg Input Tokens": "{:.1f}",
        "Latency (s)": "{:.3f}",
        "Best Generation": "{:.0f}",
        "Pareto Archive": "{:.0f}",
        "Advisor Proposals": "{:.0f}",
        "Advisor Applied": "{:.0f}",
    }, na_rep="-")
)

# ============================================================
# 4. Cloudless vs Advisor delta table
# ============================================================

delta_core_cols = [
    "label", "status",
    "cloudless_best_DETPass", "advisor_best_DETPass", "delta_best_DETPass",
    "cloudless_best_avg_DET", "advisor_best_avg_DET", "delta_best_avg_DET",
    "cloudless_tokens", "advisor_tokens", "delta_tokens",
    "advisor_proposals_generated",
    "advisor_proposals_accepted_applied",
    "advisor_children_scheduled",
]

delta_table = delta[[c for c in delta_core_cols if c in delta.columns]].copy()
display(delta_table)

print("[Mean delta DETPass]", num(delta.get("delta_best_DETPass", pd.Series(dtype=float))).mean())
print("[Mean delta Tokens]", num(delta.get("delta_tokens", pd.Series(dtype=float))).mean())

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/A6000_SET_A_fair_summary(1).csv'

In [59]:
# ============================================================
# 1. Load Pareto points from each run
# ============================================================

def load_pareto_points(row):
    out_dir = Path(str(row.get("out_dir", "")))

    candidates = [
        out_dir / "pareto_archive.csv",
        out_dir / "ga_pareto_frontier.csv",
        out_dir / "ga_pareto_generation_summary.csv",
    ]

    for p in candidates:
        if p.exists():
            df = pd.read_csv(p)
            df["run_key"] = row["run_key"]
            df["label"] = row["label"]
            df["mode"] = row["mode"]

            if "det_pass_rate" not in df.columns:
                for c in ["DETPass", "best_DETPass", "validation_det_pass_rate", "det"]:
                    if c in df.columns:
                        df["det_pass_rate"] = df[c]
                        break

            if "avg_prompt_tokens" not in df.columns:
                for c in ["accepted_best_tokens", "avg_tokens", "tokens", "prompt_tokens"]:
                    if c in df.columns:
                        df["avg_prompt_tokens"] = df[c]
                        break

            return df

    return pd.DataFrame([{
        "run_key": row["run_key"],
        "label": row["label"],
        "mode": row["mode"],
        "generation": row.get("best_generation", np.nan),
        "genome_id": row.get("best_genome_id", ""),
        "det_pass_rate": row.get("best_DETPass", np.nan),
        "avg_prompt_tokens": row.get("accepted_best_tokens", np.nan),
    }])

pareto_all = pd.concat([load_pareto_points(r) for _, r in fair.iterrows()], ignore_index=True)

pareto_all["det_pass_rate"] = num(pareto_all["det_pass_rate"])
pareto_all["avg_prompt_tokens"] = num(pareto_all["avg_prompt_tokens"])

pareto_all = pareto_all.dropna(subset=["det_pass_rate", "avg_prompt_tokens"])

# ============================================================
# 2. Global Pareto frontier: maximize DETPass, minimize tokens
# ============================================================

def global_pareto(df):
    d = df.sort_values(["avg_prompt_tokens", "det_pass_rate"], ascending=[True, False]).copy()
    best_det = -np.inf
    keep = []

    for idx, row in d.iterrows():
        if row["det_pass_rate"] > best_det:
            keep.append(idx)
            best_det = row["det_pass_rate"]

    d["is_pareto"] = d.index.isin(keep)
    return d

pareto_eval = global_pareto(pareto_all)
frontier = pareto_eval[pareto_eval["is_pareto"]].copy()

display(frontier[[
    "run_key", "label", "mode", "generation", "genome_id",
    "det_pass_rate", "avg_prompt_tokens"
]])

# ============================================================
# 3. Paper Figure-3 style plot
# ============================================================

plt.figure(figsize=(9, 6))

for (label, mode), sub in pareto_eval.groupby(["label", "mode"]):
    plt.scatter(
        sub["avg_prompt_tokens"],
        sub["det_pass_rate"],
        s=70,
        alpha=0.75,
        label=f"{label}-{mode}",
    )

if len(frontier) > 0:
    f = frontier.sort_values("avg_prompt_tokens")
    plt.plot(
        f["avg_prompt_tokens"],
        f["det_pass_rate"],
        linestyle="--",
        linewidth=2,
        label="Global Pareto frontier",
    )

for _, r in fair.iterrows():
    x = pd.to_numeric(r.get("accepted_best_tokens", np.nan), errors="coerce")
    y = pd.to_numeric(r.get("best_DETPass", np.nan), errors="coerce")
    if pd.notna(x) and pd.notna(y):
        plt.annotate(
            f"{r['label']}-{r['mode']}",
            (x, y),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9,
        )

plt.xlabel("Average Input Tokens")
plt.ylabel("DETPass (%)")
plt.title("Deployment-aware Pareto Trade-off")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ============================================================
# 4. Advisor delta plots
# ============================================================

if "delta_best_DETPass" in delta.columns:
    plt.figure(figsize=(7, 4))
    plt.bar(delta["label"], num(delta["delta_best_DETPass"]))
    plt.axhline(0, linewidth=1)
    plt.xlabel("Model")
    plt.ylabel("Advisor - Cloudless DETPass (%p)")
    plt.title("Cloud Advisor DETPass Gain")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

if "delta_tokens" in delta.columns:
    plt.figure(figsize=(7, 4))
    plt.bar(delta["label"], num(delta["delta_tokens"]))
    plt.axhline(0, linewidth=1)
    plt.xlabel("Model")
    plt.ylabel("Advisor - Cloudless Tokens")
    plt.title("Cloud Advisor Token Delta")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [66]:
# ============================================================
# 1. Load Pareto points from each run
# ============================================================

def load_pareto_points(row):
    out_dir = Path(str(row.get("out_dir", "")))

    candidates = [
        out_dir / "pareto_archive.csv",
        out_dir / "ga_pareto_frontier.csv",
        out_dir / "ga_pareto_generation_summary.csv",
    ]

    for p in candidates:
        if p.exists():
            df = pd.read_csv(p)
            df["run_key"] = row["run_key"]
            df["label"] = row["label"]
            df["mode"] = row["mode"]

            if "det_pass_rate" not in df.columns:
                for c in ["DETPass", "best_DETPass", "validation_det_pass_rate", "det"]:
                    if c in df.columns:
                        df["det_pass_rate"] = df[c]
                        break

            if "avg_prompt_tokens" not in df.columns:
                for c in ["accepted_best_tokens", "avg_tokens", "tokens", "prompt_tokens"]:
                    if c in df.columns:
                        df["avg_prompt_tokens"] = df[c]
                        break

            return df

    return pd.DataFrame([{
        "run_key": row["run_key"],
        "label": row["label"],
        "mode": row["mode"],
        "generation": row.get("best_generation", np.nan),
        "genome_id": row.get("best_genome_id", ""),
        "det_pass_rate": row.get("best_DETPass", np.nan),
        "avg_prompt_tokens": row.get("accepted_best_tokens", np.nan),
    }])

pareto_all = pd.concat([load_pareto_points(r) for _, r in fair.iterrows()], ignore_index=True)

pareto_all["det_pass_rate"] = num(pareto_all["det_pass_rate"])
pareto_all["avg_prompt_tokens"] = num(pareto_all["avg_prompt_tokens"])

pareto_all = pareto_all.dropna(subset=["det_pass_rate", "avg_prompt_tokens"])

# ============================================================
# 2. Global Pareto frontier: maximize DETPass, minimize tokens
# ============================================================

def global_pareto(df):
    d = df.sort_values(["avg_prompt_tokens", "det_pass_rate"], ascending=[True, False]).copy()
    best_det = -np.inf
    keep = []

    for idx, row in d.iterrows():
        if row["det_pass_rate"] > best_det:
            keep.append(idx)
            best_det = row["det_pass_rate"]

    d["is_pareto"] = d.index.isin(keep)
    return d

pareto_eval = global_pareto(pareto_all)
frontier = pareto_eval[pareto_eval["is_pareto"]].copy()

display(frontier[[
    "run_key", "label", "mode", "generation", "genome_id",
    "det_pass_rate", "avg_prompt_tokens"
]])

# ============================================================
# 3. Paper Figure-3 style plot
# ============================================================

plt.figure(figsize=(9, 6))

for (label, mode), sub in pareto_eval.groupby(["label", "mode"]):
    plt.scatter(
        sub["avg_prompt_tokens"],
        sub["det_pass_rate"],
        s=70,
        alpha=0.75,
        label=f"{label}-{mode}",
    )

if len(frontier) > 0:
    f = frontier.sort_values("avg_prompt_tokens")
    plt.plot(
        f["avg_prompt_tokens"],
        f["det_pass_rate"],
        linestyle="--",
        linewidth=2,
        label="Global Pareto frontier",
    )

for _, r in fair.iterrows():
    x = pd.to_numeric(r.get("accepted_best_tokens", np.nan), errors="coerce")
    y = pd.to_numeric(r.get("best_DETPass", np.nan), errors="coerce")
    if pd.notna(x) and pd.notna(y):
        plt.annotate(
            f"{r['label']}-{r['mode']}",
            (x, y),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9,
        )

plt.xlabel("Average Input Tokens")
plt.ylabel("DETPass (%)")
plt.title("Deployment-aware Pareto Trade-off")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ============================================================
# 4. Advisor delta plots
# ============================================================

if "delta_best_DETPass" in delta.columns:
    plt.figure(figsize=(7, 4))
    plt.bar(delta["label"], num(delta["delta_best_DETPass"]))
    plt.axhline(0, linewidth=1)
    plt.xlabel("Model")
    plt.ylabel("Advisor - Cloudless DETPass (%p)")
    plt.title("Cloud Advisor DETPass Gain")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

if "delta_tokens" in delta.columns:
    plt.figure(figsize=(7, 4))
    plt.bar(delta["label"], num(delta["delta_tokens"]))
    plt.axhline(0, linewidth=1)
    plt.xlabel("Model")
    plt.ylabel("Advisor - Cloudless Tokens")
    plt.title("Cloud Advisor Token Delta")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    

NameError: name 'fair' is not defined